# 05 — Soft EOS constraint and Monte Carlo stability

Notebook 03 used a deterministic EOS relation:

$$
\Lambda_1=\Lambda_{\rm EOS}(m_1;\phi_{\rm EOS}),
\qquad
\Lambda_2=\Lambda_{\rm EOS}(m_2;\phi_{\rm EOS}).
$$

Equivalently, the population model contained delta functions,

$$
p(\Lambda_1,\Lambda_2|m_1,m_2,\phi_{\rm EOS})
=
\delta\left[
\Lambda_1-\Lambda_{\rm EOS}(m_1;\phi_{\rm EOS})
\right]
\delta\left[
\Lambda_2-\Lambda_{\rm EOS}(m_2;\phi_{\rm EOS})
\right].
$$

This is the physically natural statement if all neutron stars share one universal EOS.

However, the delta-function treatment can be numerically brittle when the single-event likelihood is represented by a finite number of PE samples. The hierarchical likelihood is estimated by Monte Carlo sums over PE samples and injections. If the model places weight in a very narrow region of parameter space, only a small number of samples may contribute significantly. This produces large Monte Carlo variance.

This notebook studies a **numerical regularization** of that problem.

Instead of imposing an exact delta-function EOS constraint, we replace it by a finite-width kernel in log tidal deformability,

$$
p_{\rm soft}(\log\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\mathcal{N}
\left[
\log\tilde{\Lambda}
\middle|
\log\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}),
\sigma_{\rm soft}
\right].
$$

The strict delta-function case is recovered in the limit

$$
\sigma_{\rm soft}\rightarrow 0.
$$

## What this notebook is about

This notebook asks:

> Does replacing the exact EOS constraint by a finite-width kernel improve Monte Carlo stability?

It compares three modes:

- `strict`: use the deterministic EOS treatment from notebook 03;
- `fixed`: use a soft EOS kernel with fixed width;
- `sample`: infer the soft-kernel width and marginalize over it.


The correct interpretation is:

> The soft EOS kernel is a numerical or phenomenological regularization of the strict delta-function constraint.

At the end, we compare its impact to the model-misspecification effects seen in notebook 04.

## Colab setup

If running on Colab, first clone the repository and enter it. If running locally from the repository root, skip this cell.


In [ ]:
# # Run this cell only on Colab.
# from google.colab import drive
# drive.mount('/content/drive')

# %cd /content/drive/MyDrive
# %cd ns-eos-population-tutorial

# !pip install -q numpyro corner


## Imports and paths


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.scipy as jsp

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value

import arviz as az
import corner

from astropy.cosmology import Planck18 as cosmology
from astropy.constants import c

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data/processed"
figures_dir = project_root / "figures"
results_dir = project_root / "results"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

C_KM_S = c.to("km/s").value


## Runtime choices

The key option in this notebook is

```python
soft_eos_mode
```

It controls how the EOS constraint is implemented.

### Strict mode

```python
soft_eos_mode = "strict"
```

This reproduces the treatment from notebook 03.

The EOS relation is deterministic:

$$
\tilde{\Lambda}
=
\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}).
$$

The event tidal likelihood is evaluated at the EOS prediction,

$$
p(d_i^{\rm tidal}|m_1,m_2,\phi_{\rm EOS})
=
\mathcal{N}
\left[
\widehat{\log\tilde{\Lambda}}_i
\middle|
\log\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}),
\sigma_{\ell,i}
\right].
$$

This corresponds to a delta-function EOS population prior.

### Fixed soft mode

```python
soft_eos_mode = "fixed"
```

The exact EOS constraint is replaced by a finite-width kernel,

$$
p_{\rm soft}(\log\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\mathcal{N}
\left[
\log\tilde{\Lambda}
\middle|
\log\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}),
\sigma_{\rm soft}
\right].
$$

The width

$$
\sigma_{\rm soft}
$$

is fixed to the value set by

```python
sigma_soft_loglambda_fixed
```

This is useful if we want to test whether a modest finite width stabilizes the Monte Carlo likelihood.

### Sampled soft mode

```python
soft_eos_mode = "sample"
```

The width of the soft kernel is itself inferred:

$$
\sigma_{\rm soft}
\sim
\pi(\sigma_{\rm soft}).
$$

The posterior for $\sigma_{\rm soft}$ tells us whether the data and finite-sample likelihood prefer a nearly strict EOS constraint or a finite-width kernel.

If the posterior piles up near the lower prior bound, the strict EOS treatment is probably adequate.

If the posterior prefers a finite value, the strict delta-function treatment may be too sharp for the available PE samples or for the chosen model.

### Monte Carlo variance cut

The switch

```python
apply_hard_mc_variance_cut
```

controls whether we reject proposals with a noisy Monte Carlo likelihood estimate.

The diagnostic quantity is

$$
\sigma^2_{\log\widehat{\mathcal{L}}},
$$

the estimated variance of the log-likelihood estimator.

If the switch is off, this is only recorded as a diagnostic.

If the switch is on, proposals with

$$
\sigma^2_{\log\widehat{\mathcal{L}}} > 1
$$

are rejected.

This is not a physical prior. It is a numerical reliability cut.

In [ ]:
# These control runtime only. They do not change the generated catalog.
n_events_use = 20
n_pe_use = 1000
n_injections_use = 5_000

# NumPyro settings.
num_warmup = 200
num_samples = 200
num_chains = 2
rng_seed = 123

# Optional hard cuts. Use these to test with/without the constraints.
apply_hard_eos_physical_prior = False
apply_hard_mc_variance_cut = False

# GWTC-style Monte Carlo likelihood variance cut.
max_log_likelihood_variance = 1.0

# Soft EOS constraint options.
#   "strict": deterministic/delta-like EOS relation, as in notebook 03.
#   "fixed" : finite-width log-normal EOS kernel with fixed sigma.
#   "sample": finite-width log-normal EOS kernel with sigma sampled/marginalized.
soft_eos_mode = "sample"
sigma_soft_loglambda_fixed = 0.10
sigma_soft_loglambda_prior = ("uniform", 0.01, 0.50)
sigma_soft_loglambda_init = 0.10


## Inputs from previous notebooks

This notebook uses the same data products as notebook 03:

1. The EOS polynomial fit from notebook 00.

2. The intrinsic population from notebook 01.

3. The mock detected events and PE samples from notebook 02.

4. The detected injections from notebook 02.

The important point is that we do not generate a new population or a new catalog here. We reuse the same mock data so that any differences in the posterior can be attributed to the treatment of the EOS constraint.

The comparison is therefore:

$$
\text{same mock PE samples}
+
\text{same injections}
+
\text{different EOS constraint}.
$$

This isolates the numerical effect of replacing the strict delta function by a finite-width kernel.

In [ ]:
# Tagged inputs from notebooks 00--02.
# Edit these filenames if you changed the upstream tags.

eos_fit_filename = "eos_fit__bsk24_m1p00_2p25_poly5.npz"
population_filename = "intrinsic_population__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993.npz"
detected_filename = "detected_events__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871.npz"
pe_filename = "mock_pe_samples__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871.npz"
injections_filename = "injections__bsk24_m1p00_2p25_poly5__smoothbox_md_z20_n100000_seed31031993__det_rho80_thr8_nev100_pe5000_seed1871__inj_n200000_z20_edge0p001_seed1871.npz"


eos_fit = np.load(processed_dir / eos_fit_filename)
intrinsic = np.load(processed_dir / population_filename)
detected = np.load(processed_dir / detected_filename)
pe = np.load(processed_dir / pe_filename)
injections = np.load(processed_dir / injections_filename)

def read_npz_string(data, key):
    value = data[key]
    if np.asarray(value).shape == ():
        return str(value.item())
    return str(value)

eos_tag = read_npz_string(eos_fit, "eos_tag")
eos_base_tag = read_npz_string(eos_fit, "eos_base_tag") if "eos_base_tag" in eos_fit else ""
fit_mass_tag = read_npz_string(eos_fit, "fit_mass_tag") if "fit_mass_tag" in eos_fit else ""
pop_tag = read_npz_string(intrinsic, "pop_tag")
detpe_tag = read_npz_string(detected, "detpe_tag")
inj_tag = read_npz_string(injections, "inj_tag")



n_events_available = detected["m1_det"].shape[0]
n_pe_available = pe["m1_det"].shape[1]
n_injections_available = injections["m1_det"].shape[0]
n_inj_drawn_total = int(injections["n_inj_drawn"])


if n_injections_use==-1:
    n_injections_use = n_injections_available


def number_tag(x):
    x = float(x)
    if abs(x - round(x)) < 1e-12:
        return str(int(round(x)))
    return (f"{x:.3g}").replace(".", "p").replace("-", "m")

if soft_eos_mode == "strict":
    soft_tag = "soft_strict"
elif soft_eos_mode == "fixed":
    soft_tag = f"soft_fixedsig{number_tag(sigma_soft_loglambda_fixed)}"
elif soft_eos_mode == "sample":
    soft_tag = (
        "soft_samplesig"
        f"{number_tag(sigma_soft_loglambda_prior[1])}_{number_tag(sigma_soft_loglambda_prior[2])}"
    )
else:
    raise ValueError(f"Unknown soft_eos_mode: {soft_eos_mode}")

inf_tag = (
    f"inf_nev{n_events_use}_npe{n_pe_use}_ninj{n_injections_use}"
    f"_w{num_warmup}_s{num_samples}_c{num_chains}_seed{rng_seed}"
)
run_tag = f"{eos_tag}__{pop_tag}__{detpe_tag}__{inj_tag}__{soft_tag}__{inf_tag}"

fit_min = float(eos_fit["fit_min"])
fit_max = float(eos_fit["fit_max"])
eos_order = int(eos_fit["chosen_order"])
eos_coeffs_fit = np.array(eos_fit["chosen_coeffs"])

edge_pdf_value = float(intrinsic["edge_pdf_value"])

print(f"EOS tag: {eos_tag}")
print(f"Population tag: {pop_tag}")
print(f"Detection/PE tag: {detpe_tag}")
print(f"Injection tag: {inj_tag}")
print(f"Soft-EOS tag: {soft_tag}")
print(f"Inference tag: {inf_tag}")
print(f"Run tag: {run_tag}")
print()
print(f"Available events: {n_events_available}")
print(f"Available PE samples per event: {n_pe_available}")
print(f"Available detected injections: {n_injections_available}")
print(f"Total injections drawn in notebook 02: {n_inj_drawn_total}")
print(f"EOS order: {eos_order}")
print("EOS coefficients, highest power first:")
print(eos_coeffs_fit)


## Subsampling for fast runs

As in notebook 03, we may use only a subset of events, PE samples, and detected injections.

Reducing these numbers makes the notebook faster but increases Monte Carlo noise.

This is especially relevant in this notebook because the whole point is to diagnose Monte Carlo stability. A small number of PE samples can make the strict EOS constraint look unstable, while a larger number of samples may reduce the problem.

The key quantities are:

- `n_events_use`: number of detected events used;
- `n_pe_use`: number of PE samples used per event;
- `n_injections_use`: number of detected injections used for the selection correction.

If the soft kernel appears to have a large effect, one useful exercise is to rerun the notebook with larger `n_pe_use` or `n_injections_use` and check whether the effect decreases.

In [ ]:
rng = np.random.default_rng(rng_seed)


if n_events_use > n_events_available:
    raise ValueError(f"n_events_use={n_events_use} exceeds available events={n_events_available}")
if n_pe_use > n_pe_available:
    raise ValueError(f"n_pe_use={n_pe_use} exceeds available PE samples per event={n_pe_available}")
if n_injections_use > n_injections_available:
    raise ValueError(f"n_injections_use={n_injections_use} exceeds available detected injections={n_injections_available}")

event_idx = np.arange(n_events_available)[:n_events_use]
pe_idx = rng.choice(n_pe_available, size=n_pe_use, replace=False)
inj_idx = rng.choice(n_injections_available, size=n_injections_use, replace=False)

pe_m1_det = pe["m1_det"][event_idx][:, pe_idx]
pe_m2_det = pe["m2_det"][event_idx][:, pe_idx]
pe_d_l = pe["d_l"][event_idx][:, pe_idx]
pe_lambda_tilde = pe["lambda_tilde"][event_idx][:, pe_idx]
pe_log_lambda_tilde = np.log(np.maximum(pe_lambda_tilde, 1e-300))

obs_log_lambdatilde = pe["obs_log_lambdatilde"][event_idx]
sigma_log_lambdatilde = pe["sigma_log_lambdatilde"][event_idx]

inj_m1_det = injections["m1_det"][inj_idx]
inj_m2_det = injections["m2_det"][inj_idx]
inj_d_l = injections["d_l"][inj_idx]
inj_log_p_draw = injections["log_p_draw"][inj_idx]

# We use only a random subset of found injections.  The selection estimator
# must therefore estimate the full found-injection sum by multiplying the
# subset sum by this factor.
n_inj_drawn = n_inj_drawn_total
inj_subset_factor = n_injections_available / n_injections_use

print(f"Using events: {n_events_use}")
print(f"Using PE samples per event: {n_pe_use}")
print(f"Using detected injections: {n_injections_use}")
print(f"Detected-injection subset factor: {inj_subset_factor:.6g}")
print(f"Total injections drawn in notebook 02: {n_inj_drawn}")


## Cosmology grid

The PE samples and injections are stored in detector-frame variables,

$
m_{1,z},\quad m_{2,z},\quad d_L .
$

The population model is defined in source-frame variables,

$
m_1,\quad m_2,\quad z .
$

For fixed cosmology we precompute a grid for

$
d_L(z),\quad z(d_L),\quad \log \frac{dV_c}{dz},\quad \log \frac{dd_L}{dz}.
$

The source-to-detector Jacobian is

$
\left|
\frac{\partial(m_1,m_2,z)}
{\partial(m_{1,z},m_{2,z},d_L)}
\right| = \frac{1}{(1+z)^2}
\frac{1}{dd_L/dz}.
$

So

$
\log p_{\rm pop}^{\rm det} = \log p_{\rm pop}^{\rm src}
-2\log(1+z)
-\log(dd_L/dz).
$


In [ ]:
z_grid_np = np.linspace(1e-6, 25.0, 6000)
d_l_grid_np = cosmology.luminosity_distance(z_grid_np).value
dc_grid_np = cosmology.comoving_distance(z_grid_np).value
Hz_grid_np = cosmology.H(z_grid_np).value

log_dV_dz_grid_np = np.log(4.0 * np.pi) + np.log(C_KM_S) - np.log(Hz_grid_np) + 2.0 * np.log(dc_grid_np)
ddL_dz_grid_np = dc_grid_np + (1.0 + z_grid_np) * C_KM_S / Hz_grid_np
log_ddL_dz_grid_np = np.log(ddL_dz_grid_np)

z_grid = jnp.asarray(z_grid_np)
d_l_grid = jnp.asarray(d_l_grid_np)
log_dV_dz_grid = jnp.asarray(log_dV_dz_grid_np)
log_ddL_dz_grid = jnp.asarray(log_ddL_dz_grid_np)

def z_from_d_l_jax(d_l):
    return jnp.interp(d_l, d_l_grid, z_grid)

def interp_log_dV_dz(z):
    return jnp.interp(z, z_grid, log_dV_dz_grid)

def interp_log_ddL_dz(z):
    return jnp.interp(z, z_grid, log_ddL_dz_grid)


## Population model

The mass model is the same smoothed-box family used for the injections,

$
p(m)\propto
\sigma\left(\frac{m-m_{\min}-w}{s}\right)
\sigma\left(\frac{m_{\max}-w-m}{s}\right),
$

normalized on the EOS-valid interval.

The redshift model is

$
p(z|\alpha_z,\beta_z,z_p)
\propto
\frac{R(z|\alpha_z,\beta_z,z_p)}{1+z}
\frac{dV_c}{dz},
$

with

$
R(z)
\propto
\frac{(1+z)^{\alpha_z}}
{1+\left[(1+z)/(1+z_p)\right]^{\alpha_z+\beta_z}}.
$

The component masses are independently drawn and then sorted. For ordered masses $m_1\ge m_2$,

$
p(m_1,m_2)=2p(m_1)p(m_2).
$


In [ ]:
mass_grid = jnp.linspace(fit_min, fit_max, 1000)

def sigmoid_jax(x):
    return jax.nn.sigmoid(x)

def bounded_smooth_mass_unnorm(m, m_min, m_max, edge_width):
    logit_edge = jnp.log((1.0 - edge_pdf_value) / edge_pdf_value)
    edge_scale = edge_width / logit_edge
    left = sigmoid_jax((m - m_min - edge_width) / edge_scale)
    right = sigmoid_jax((m_max - edge_width - m) / edge_scale)
    return left * right

def log_mass_pdf(m, m_min, m_max, edge_width):
    pdf_grid = bounded_smooth_mass_unnorm(mass_grid, m_min, m_max, edge_width)
    norm = jnp.trapezoid(pdf_grid, mass_grid)

    pdf = bounded_smooth_mass_unnorm(m, m_min, m_max, edge_width) / norm

    in_support = (m >= fit_min) & (m <= fit_max)
    return jnp.where(
        in_support,
        jnp.log(jnp.maximum(pdf, 1e-300)),
        -1e30,
    )

def merger_rate_density_jax(z, alpha_z, beta_z, z_p):
    numerator = (1.0 + z) ** alpha_z
    denominator = 1.0 + ((1.0 + z) / (1.0 + z_p)) ** (alpha_z + beta_z)
    return numerator / denominator

def log_redshift_pdf(z, alpha_z, beta_z, z_p):
    rate_grid = merger_rate_density_jax(z_grid, alpha_z, beta_z, z_p)
    integrand_grid = rate_grid * jnp.exp(log_dV_dz_grid) / (1.0 + z_grid)
    norm = jnp.trapezoid(integrand_grid, z_grid)

    log_rate = jnp.log(merger_rate_density_jax(z, alpha_z, beta_z, z_p))
    return log_rate + interp_log_dV_dz(z) - jnp.log1p(z) - jnp.log(norm)

def log_pop_source(m1, m2, z, m_min, m_max, edge_width, alpha_z, beta_z, z_p):
    return (
        jnp.log(2.0)
        + log_mass_pdf(m1, m_min, m_max, edge_width)
        + log_mass_pdf(m2, m_min, m_max, edge_width)
        + log_redshift_pdf(z, alpha_z, beta_z, z_p)
    )


## EOS model and tidal constraint

This is the central conceptual change in notebook 05.

In notebook 03, the EOS relation was deterministic. For each PE sample with source-frame masses $(m_1,m_2)$ and proposed EOS parameters $\phi_{\rm EOS}$, the model predicted

$$
\tilde{\Lambda}_{\rm EOS}
=
\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}).
$$

The tidal part of the event likelihood was

$$
p(d_i^{\rm tidal}|m_1,m_2,\phi_{\rm EOS})
=
\mathcal{N}
\left[
\widehat{\ell}_i
\middle|
\log\tilde{\Lambda}_{\rm EOS},
\sigma_{\ell,i}
\right],
$$

where

$$
\widehat{\ell}_i=\widehat{\log\tilde{\Lambda}}_i.
$$

This corresponds to the strict population prior

$$
p(\log\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS})
=
\delta\left[
\log\tilde{\Lambda}
-
\log\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS})
\right].
$$

In this notebook we replace the delta function with a Gaussian kernel,

$$
p_{\rm soft}(\log\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\mathcal{N}
\left[
\log\tilde{\Lambda}
\middle|
\log\tilde{\Lambda}_{\rm EOS}(m_1,m_2;\phi_{\rm EOS}),
\sigma_{\rm soft}
\right].
$$

The event likelihood now marginalizes over the latent true value of $\log\tilde{\Lambda}$:

$$
p(d_i^{\rm tidal}|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\int d\ell\,
p(d_i^{\rm tidal}|\ell)
p_{\rm soft}(\ell|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft}),
$$

where

$$
\ell = \log\tilde{\Lambda}.
$$

The mock tidal likelihood is

$$
p(d_i^{\rm tidal}|\ell)
=
\mathcal{N}
\left[
\widehat{\ell}_i
\middle|
\ell,
\sigma_{\ell,i}
\right].
$$

The soft EOS kernel is

$$
p_{\rm soft}(\ell|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\mathcal{N}
\left[
\ell
\middle|
\log\tilde{\Lambda}_{\rm EOS},
\sigma_{\rm soft}
\right].
$$
This is a finite-width numerical/model-discrepancy kernel, not a statement that different events have independent physical EOSs.

Because the mock detection model does not depend on tidal deformability, the selection term is left unchanged. In a more general analysis where detection/selection depends on tidal parameters, injections would need to cover the tidal coordinate and the injection draw density would have to include it.

In [ ]:
def polyval_jax(coeffs, x):
    y = jnp.zeros_like(x)
    for c in coeffs:
        y = y * x + c
    return y

def lambda_of_mass_jax(m, coeffs):
    return jnp.exp(polyval_jax(coeffs, m))

def lambda_tilde_jax(m1, m2, lambda1, lambda2):
    return (16.0 / 13.0) * (
        (m1 + 12.0 * m2) * m1**4 * lambda1
        + (m2 + 12.0 * m1) * m2**4 * lambda2
    ) / (m1 + m2) ** 5

def log_tidal_likelihood(log_lambda_tilde_eos, obs_log_lambdatilde, sigma):
    return dist.Normal(obs_log_lambdatilde[:, None], sigma[:, None]).log_prob(log_lambda_tilde_eos)


def log_normal_pdf_jax(x, loc, scale):
    return dist.Normal(loc, scale).log_prob(x)


## Parameter configuration

The sampled hyperparameters are the same as in notebook 03:

$$
m_{\min},\quad
m_{\max},\quad
\alpha_z,\quad
\beta_z,\quad
z_p,\quad
a_0,\ldots,a_5.
$$

These describe the mass distribution, redshift evolution, and EOS polynomial.

If

```python
soft_eos_mode = "sample"
```

then the model also samples

$$
\sigma_{\rm soft}.
$$

This parameter controls the width of the soft EOS kernel in log tidal deformability.


A useful interpretation is:

- if $\sigma_{\rm soft}$ is driven to the lower prior bound, the strict EOS constraint is adequate;
- if $\sigma_{\rm soft}$ is inferred to be finite, the data or Monte Carlo representation prefer a softened constraint;
- if $\sigma_{\rm soft}$ becomes very large, the model may be washing out the tidal information.

In [ ]:
m_min_true = float(intrinsic["m_min_true"])
m_max_true = float(intrinsic["m_max_true"])
edge_width_true = float(intrinsic["edge_width_true"])
alpha_z_true = float(intrinsic["alpha_z"])
beta_z_true = float(intrinsic["beta_z"])
z_p_true = float(intrinsic["z_p"])

param_config = {
    "m_min": {
        "sample": True,
        "fixed": m_min_true,
        "prior": ("uniform", fit_min - 0.05, fit_min + 0.05),
        "init": m_min_true,
    },
    "m_max": {
        "sample": True,
        "fixed": m_max_true,
        "prior": ("uniform", fit_max - 0.05, fit_max + 0.05),
        "init": m_max_true,
    },
    "edge_width": {
        "sample": False,
        "fixed": edge_width_true,
        "prior": ("uniform", 0.01, 0.15),
        "init": edge_width_true,
    },
    "alpha_z": {
        "sample": True,
        "fixed": alpha_z_true,
        "prior": ("uniform", 1.0, 5.0),
        "init": alpha_z_true,
    },
    "beta_z": {
        "sample": True,
        "fixed": beta_z_true,
        "prior": ("uniform", 1.0, 5.0),
        "init": beta_z_true,
    },
    "z_p": {
        "sample": True,
        "fixed": z_p_true,
        "prior": ("uniform", 1.0, 3.0),
        "init": z_p_true,
    },
}

for k, coeff in enumerate(eos_coeffs_fit):
    width = 0.05 * abs(float(coeff)) + 0.01
    param_config[f"eos_a{k}"] = {
        "sample": True,
        "fixed": float(coeff),
        "prior": ("normal", float(coeff), width),
        "init": float(coeff),
    }


# Width of the soft EOS kernel. It is sampled only in soft_eos_mode == "sample".
if soft_eos_mode == "sample":
    param_config["sigma_soft_loglambda"] = {
        "sample": True,
        "fixed": sigma_soft_loglambda_fixed,
        "prior": sigma_soft_loglambda_prior,
        "init": sigma_soft_loglambda_init,
    }
elif soft_eos_mode == "fixed":
    param_config["sigma_soft_loglambda"] = {
        "sample": False,
        "fixed": sigma_soft_loglambda_fixed,
        "prior": sigma_soft_loglambda_prior,
        "init": sigma_soft_loglambda_fixed,
    }

for name, cfg in param_config.items():
    print(name, cfg)


In [ ]:
# Prior predictive check for the EOS function, with physical prior support

n_prior_draws = 1000
n_plot_draws = 200
m_plot = np.linspace(fit_min, fit_max, 300)

def eos_is_valid_np(coeffs, m_grid):
    log_lambda = np.polyval(coeffs, m_grid)

    valid = (
        np.all(np.isfinite(log_lambda))
        and np.all(log_lambda > np.log(0.1))
        and np.all(log_lambda < np.log(1e5))
        and np.all(np.diff(log_lambda) < 0.0)
    )
    return valid

valid_curves = []
valid_coeffs = []

n_try = 0
max_tries = 20000

while len(valid_curves) < n_prior_draws and n_try < max_tries:
    coeff_draw = []

    for k, coeff in enumerate(eos_coeffs_fit):
        cfg = param_config[f"eos_a{k}"]
        prior = cfg["prior"]

        if cfg["sample"]:
            if prior[0] == "normal":
                coeff_draw.append(rng.normal(prior[1], prior[2]))
            elif prior[0] == "uniform":
                coeff_draw.append(rng.uniform(prior[1], prior[2]))
            else:
                raise ValueError(prior[0])
        else:
            coeff_draw.append(cfg["fixed"])

    coeff_draw = np.array(coeff_draw)

    if eos_is_valid_np(coeff_draw, m_plot):
        lambda_curve = np.exp(np.polyval(coeff_draw, m_plot))
        valid_curves.append(lambda_curve)
        valid_coeffs.append(coeff_draw)

    n_try += 1

valid_curves = np.array(valid_curves)
valid_coeffs = np.array(valid_coeffs)

print(f"Accepted {len(valid_curves)} valid EOS prior draws out of {n_try} attempts")

injected_lambda = np.exp(np.polyval(eos_coeffs_fit, m_plot))

fig, ax = plt.subplots(figsize=(7.0, 4.5))

n_show = min(n_plot_draws, len(valid_curves))
show_idx = rng.choice(len(valid_curves), size=n_show, replace=False)

for idx in show_idx:
    ax.plot(m_plot, valid_curves[idx], lw=0.8, alpha=0.3)

ax.plot(m_plot, injected_lambda, color="black", lw=2.0, label="Injected polynomial")

ax.set_yscale("log")
ax.set_ylim(0.1, 1e05)
ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$\Lambda(m)$")
ax.set_title("EOS prior predictive (physical prior support)")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
prior_eos_path = figures_dir / f"eos_prior_predictive__{run_tag}.png"
fig.savefig(prior_eos_path, dpi=200)
plt.show()

print(f"Saved EOS prior predictive plot to {prior_eos_path}")

## Hierarchical likelihood and Monte Carlo stability

The hierarchical likelihood is the same rate-marginalized likelihood as in notebook 03,

$$
\log\widehat{\mathcal{L}}(\lambda)
=
\sum_i
\log\widehat{p}(d_i|\lambda)
-
N_{\rm det}\log\widehat{\xi}(\lambda).
$$

The event contribution is estimated from PE samples,

$$
\widehat{p}(d_i|\lambda)
=
\frac{1}{N_{\rm PE}}
\sum_{j=1}^{N_{\rm PE}}
\frac{
p_{\rm pop}^{\rm det}(x_{ij}|\lambda)
}{
\pi_{\rm PE}(x_{ij})
}
p(d_i^{\rm tidal}|x_{ij},\phi_{\rm EOS},\sigma_{\rm soft}).
$$

The selection efficiency is estimated from detected injections,

$$
\widehat{\xi}(\lambda)
=
\frac{1}{N_{\rm draw}}
\sum_{j\in{\rm found}}
\frac{
p_{\rm pop}^{\rm det}(\theta_j|\lambda)
}{
p_{\rm draw}^{\rm det}(\theta_j)
}.
$$

The selection term is not directly changed by the soft tidal kernel, because the injections are used to estimate detectability of the mass-redshift population. The main change is in the event tidal likelihood.

## Why softening can improve Monte Carlo stability

The strict EOS constraint can make the event weights highly uneven. A small number of PE samples may receive most of the weight if they happen to lie close to the EOS-predicted tidal relation.

This increases the Monte Carlo variance of the event likelihood estimator.

The soft kernel broadens the tidal factor,

$$
\sigma_{\rm eff,i}^2
=
\sigma_{\ell,i}^2+\sigma_{\rm soft}^2,
$$

so more PE samples can contribute non-negligible weight. This can reduce the variance of

$$
\widehat{p}(d_i|\lambda).
$$

However, this comes at a cost: the tidal information is effectively weakened. The posterior can become broader or shift if the soft width is large.

The diagnostic question is therefore:

> Does softening mainly reduce numerical variance, or does it materially change the inferred EOS and population parameters?

In [ ]:
pe_m1_det_j = jnp.asarray(pe_m1_det)
pe_m2_det_j = jnp.asarray(pe_m2_det)
pe_d_l_j = jnp.asarray(pe_d_l)
pe_log_lambda_tilde_j = jnp.asarray(pe_log_lambda_tilde)

obs_log_lambdatilde_j = jnp.asarray(obs_log_lambdatilde)
sigma_log_lambdatilde_j = jnp.asarray(sigma_log_lambdatilde)

inj_m1_det_j = jnp.asarray(inj_m1_det)
inj_m2_det_j = jnp.asarray(inj_m2_det)
inj_d_l_j = jnp.asarray(inj_d_l)
inj_log_p_draw_j = jnp.asarray(inj_log_p_draw)

def sample_parameter(name, cfg):
    prior = cfg["prior"]
    if cfg["sample"]:
        if prior[0] == "uniform":
            return numpyro.sample(name, dist.Uniform(prior[1], prior[2]))
        if prior[0] == "normal":
            return numpyro.sample(name, dist.Normal(prior[1], prior[2]))
        raise ValueError(f"Unknown prior type for {name}: {prior[0]}")
    return jnp.asarray(cfg["fixed"])

def relative_variance_from_logweights(logw, axis):
    max_logw = jnp.max(logw, axis=axis, keepdims=True)
    w = jnp.exp(logw - max_logw)

    n = logw.shape[axis]
    mean_w = jnp.mean(w, axis=axis, keepdims=True)
    var_mean = jnp.sum((w - mean_w) ** 2, axis=axis) / (n * (n - 1))
    mean_w_flat = jnp.squeeze(mean_w, axis=axis)

    return var_mean / (mean_w_flat ** 2)

def logmeanexp(logw, axis):
    return jsp.special.logsumexp(logw, axis=axis) - jnp.log(logw.shape[axis])



def soft_barrier_lower(x, xmin, width):
    return -jax.nn.softplus((xmin - x) / width)

def soft_barrier_upper(x, xmax, width):
    return -jax.nn.softplus((x - xmax) / width)

    

@jax.jit
def compute_hierarchical_loglike(
    m_min,
    m_max,
    edge_width,
    alpha_z,
    beta_z,
    z_p,
    coeffs,
    sigma_soft_loglambda,
):
    pe_z = z_from_d_l_jax(pe_d_l_j)
    pe_m1 = pe_m1_det_j / (1.0 + pe_z)
    pe_m2 = pe_m2_det_j / (1.0 + pe_z)

    pe_log_pop_src = log_pop_source(
        pe_m1, pe_m2, pe_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )

    pe_log_jac = -2.0 * jnp.log1p(pe_z) - interp_log_ddL_dz(pe_z)
    pe_log_pe_prior = 2.0 * jnp.log(pe_d_l_j)

    pe_lambda1 = lambda_of_mass_jax(pe_m1, coeffs)
    pe_lambda2 = lambda_of_mass_jax(pe_m2, coeffs)
    pe_lambda_tilde_eos = lambda_tilde_jax(pe_m1, pe_m2, pe_lambda1, pe_lambda2)
    pe_log_lambda_tilde_eos = jnp.log(jnp.maximum(pe_lambda_tilde_eos, 1e-300))

    if soft_eos_mode == "strict":
        # Delta-like treatment from notebook 03.  The PE tidal coordinate is
        # evaluated analytically through the Gaussian mock likelihood.
        log_tidal_factor = log_tidal_likelihood(
            pe_log_lambda_tilde_eos,
            obs_log_lambdatilde_j,
            sigma_log_lambdatilde_j,
        )
    else:
        # Soft EOS kernel in the population factor, evaluated directly on the
        # PE samples in log LambdaTilde.
        log_tidal_factor = log_normal_pdf_jax(
            pe_log_lambda_tilde_j,
            pe_log_lambda_tilde_eos,
            sigma_soft_loglambda,
        )

    log_event_weights = pe_log_pop_src + pe_log_jac - pe_log_pe_prior + log_tidal_factor
    log_L_events = logmeanexp(log_event_weights, axis=1)
    event_rel_var = relative_variance_from_logweights(log_event_weights, axis=1)

    inj_z = z_from_d_l_jax(inj_d_l_j)
    inj_m1 = inj_m1_det_j / (1.0 + inj_z)
    inj_m2 = inj_m2_det_j / (1.0 + inj_z)

    inj_log_pop_src = log_pop_source(
        inj_m1, inj_m2, inj_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )
    inj_log_jac = -2.0 * jnp.log1p(inj_z) - interp_log_ddL_dz(inj_z)
    inj_log_pop_det = inj_log_pop_src + inj_log_jac

    log_inj_weights = inj_log_pop_det - inj_log_p_draw_j

    max_logW = jnp.max(log_inj_weights)
    W_scaled = jnp.exp(log_inj_weights - max_logW)

    sum_W_scaled = jnp.sum(W_scaled)
    sum_W2_scaled = jnp.sum(W_scaled ** 2)

    # Correct for using a random subset of the found injections.
    # The full found-injection sum is estimated as
    # inj_subset_factor times the subset sum.
    xi_scaled = inj_subset_factor * sum_W_scaled / n_inj_drawn
    log_xi = max_logW + jnp.log(inj_subset_factor * sum_W_scaled) - jnp.log(n_inj_drawn)

    # Same correction for the second moment entering the GWTC-style MC variance.
    var_xi_scaled = (
        inj_subset_factor * sum_W2_scaled - n_inj_drawn * xi_scaled ** 2
    ) / (n_inj_drawn * (n_inj_drawn - 1))
    var_xi_scaled = jnp.maximum(var_xi_scaled, 0.0)
    xi_rel_var = var_xi_scaled / (xi_scaled ** 2)

    n_det = pe_m1_det_j.shape[0]
    log_likelihood_variance = jnp.sum(event_rel_var) + n_det**2 * xi_rel_var

    log_likelihood = jnp.sum(log_L_events) - n_det * log_xi

    return (
        log_likelihood,
        log_xi,
        xi_rel_var,
        jnp.sum(event_rel_var),
        log_likelihood_variance,
    )



def hierarchical_model():
    m_min = sample_parameter("m_min", param_config["m_min"])
    m_max = sample_parameter("m_max", param_config["m_max"])
    edge_width = sample_parameter("edge_width", param_config["edge_width"])
    alpha_z = sample_parameter("alpha_z", param_config["alpha_z"])
    beta_z = sample_parameter("beta_z", param_config["beta_z"])
    z_p = sample_parameter("z_p", param_config["z_p"])

    coeffs = []
    for k in range(eos_order + 1):
        coeffs.append(sample_parameter(f"eos_a{k}", param_config[f"eos_a{k}"]))
    coeffs = jnp.stack(coeffs)

    if soft_eos_mode == "strict":
        sigma_soft_loglambda = jnp.asarray(0.0)
    else:
        sigma_soft_loglambda = sample_parameter(
            "sigma_soft_loglambda",
            param_config["sigma_soft_loglambda"],
        )

    numpyro.factor("mass_order_constraint", jnp.where(m_min < m_max, 0.0, -jnp.inf))

    log_lambda_grid = polyval_jax(coeffs, mass_grid)

    # quick version - makes divergences appear
    eos_valid = (
        jnp.all(jnp.isfinite(log_lambda_grid))
        & jnp.all(log_lambda_grid > jnp.log(0.1))
        & jnp.all(log_lambda_grid < jnp.log(1e5))
        & jnp.all(jnp.diff(log_lambda_grid) < 0.0)
    )
    
    numpyro.deterministic("eos_is_physical", eos_valid)
    if apply_hard_eos_physical_prior:
        numpyro.factor("eos_physical_prior", jnp.where(eos_valid, 0.0, -1e30))

    # lengty version - smooths boundaries
    # log_lambda_min = jnp.log(0.1)
    # log_lambda_max = jnp.log(1e5)
    
    # eos_barrier_width = 0.05
    
    # log_lambda_lower_penalty = jnp.sum(
    #     soft_barrier_lower(log_lambda_grid, log_lambda_min, eos_barrier_width)
    # )
    
    # log_lambda_upper_penalty = jnp.sum(
    #     soft_barrier_upper(log_lambda_grid, log_lambda_max, eos_barrier_width)
    # )
    
    # monotonicity_penalty = jnp.sum(
    #     -jax.nn.softplus(jnp.diff(log_lambda_grid) / eos_barrier_width)
    # )
    
    # numpyro.factor(
    #     "eos_physical_prior",
    #     log_lambda_lower_penalty
    #     + log_lambda_upper_penalty
    #     + monotonicity_penalty,
    # )

    (
        log_likelihood,
        log_xi,
        selection_rel_var,
        sum_event_rel_var,
        log_likelihood_variance,
    ) = compute_hierarchical_loglike(
        m_min,
        m_max,
        edge_width,
        alpha_z,
        beta_z,
        z_p,
        coeffs,
        sigma_soft_loglambda,
        )

    numpyro.deterministic("log_xi", log_xi)
    numpyro.deterministic("selection_rel_var", selection_rel_var)
    numpyro.deterministic("sum_event_rel_var", sum_event_rel_var)
    numpyro.deterministic("log_likelihood_variance", log_likelihood_variance)
    numpyro.deterministic("log_likelihood_value", log_likelihood)

    numpyro.deterministic(
        "passes_mc_variance_cut",
        log_likelihood_variance < max_log_likelihood_variance,
        )

    if apply_hard_mc_variance_cut:
        numpyro.factor("variance_cut", jnp.where(log_likelihood_variance < max_log_likelihood_variance, 0.0, -1e30))

    # mc_variance_barrier_width = 0.1

    # mc_variance_penalty = -jax.nn.softplus(
    #     (log_likelihood_variance - max_log_likelihood_variance)
    #     / mc_variance_barrier_width
    # )
    
    # numpyro.factor("mc_variance_cut", mc_variance_penalty)

    numpyro.factor("hierarchical_likelihood", log_likelihood)

## Run NumPyro

We now sample the posterior distribution for the chosen soft-EOS mode.

The target distribution is

$$
p(\lambda|\{d_i\})
\propto
\pi(\lambda)
\exp\left[
\log\widehat{\mathcal{L}}(\lambda)
\right].
$$

If `soft_eos_mode = "sample"`, the sampled parameter vector also includes $\sigma_{\rm soft}$.

When comparing strict and soft runs, keep the random seed, number of events, number of PE samples, and number of injections fixed as much as possible. Otherwise it becomes difficult to tell whether differences are due to the soft EOS constraint or due to Monte Carlo noise from different subsamples.

If the strict run has large Monte Carlo variance or poor effective sample size, and the soft run does not, that suggests the delta-function EOS constraint was numerically too sharp for the available PE samples.

If the soft run gives very different EOS curves or population parameters, then the softening is not merely a numerical convenience; it is changing the inference.

In [ ]:
init_values = {
    name: cfg["init"]
    for name, cfg in param_config.items()
    if cfg["sample"]
}

#### Check finiteness of initial point; useful for debugging. Not indispensable

In [ ]:
def debug_likelihood_at_values(values):
    m_min = values["m_min"]
    m_max = values["m_max"]
    edge_width = values["edge_width"]
    alpha_z = values["alpha_z"]
    beta_z = values["beta_z"]
    z_p = values["z_p"]

    coeffs = jnp.array([values[f"eos_a{k}"] for k in range(eos_order + 1)])

    pe_z = z_from_d_l_jax(pe_d_l_j)
    pe_m1 = pe_m1_det_j / (1.0 + pe_z)
    pe_m2 = pe_m2_det_j / (1.0 + pe_z)

    pe_log_pop_src = log_pop_source(
        pe_m1, pe_m2, pe_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )

    pe_log_jac = -2.0 * jnp.log1p(pe_z) - interp_log_ddL_dz(pe_z)
    pe_log_pe_prior = 2.0 * jnp.log(pe_d_l_j)

    pe_lambda1 = lambda_of_mass_jax(pe_m1, coeffs)
    pe_lambda2 = lambda_of_mass_jax(pe_m2, coeffs)
    pe_lambda_tilde_eos = lambda_tilde_jax(pe_m1, pe_m2, pe_lambda1, pe_lambda2)
    pe_log_lambda_tilde_eos = jnp.log(pe_lambda_tilde_eos)

    if soft_eos_mode == "strict":
        log_tidal_factor = log_tidal_likelihood(
            pe_log_lambda_tilde_eos,
            obs_log_lambdatilde_j,
            sigma_log_lambdatilde_j,
        )
    else:
        sigma_soft_loglambda = values.get(
            "sigma_soft_loglambda",
            sigma_soft_loglambda_fixed,
        )
        log_tidal_factor = log_normal_pdf_jax(
            pe_log_lambda_tilde_j,
            pe_log_lambda_tilde_eos,
            sigma_soft_loglambda,
        )

    log_event_weights = pe_log_pop_src + pe_log_jac - pe_log_pe_prior + log_tidal_factor
    log_L_events = logmeanexp(log_event_weights, axis=1)

    inj_z = z_from_d_l_jax(inj_d_l_j)
    inj_m1 = inj_m1_det_j / (1.0 + inj_z)
    inj_m2 = inj_m2_det_j / (1.0 + inj_z)

    inj_log_pop_src = log_pop_source(
        inj_m1, inj_m2, inj_z,
        m_min, m_max, edge_width,
        alpha_z, beta_z, z_p,
    )

    inj_log_jac = -2.0 * jnp.log1p(inj_z) - interp_log_ddL_dz(inj_z)
    inj_log_pop_det = inj_log_pop_src + inj_log_jac

    log_inj_weights = inj_log_pop_det - inj_log_p_draw_j

    quantities = {
        "pe_z": pe_z,
        "pe_m1": pe_m1,
        "pe_m2": pe_m2,
        "pe_log_pop_src": pe_log_pop_src,
        "pe_log_jac": pe_log_jac,
        "pe_log_pe_prior": pe_log_pe_prior,
        "pe_lambda1": pe_lambda1,
        "pe_lambda2": pe_lambda2,
        "pe_lambda_tilde_eos": pe_lambda_tilde_eos,
        "pe_log_lambda_tilde_eos": pe_log_lambda_tilde_eos,
        "log_tidal_factor": log_tidal_factor,
        "log_event_weights": log_event_weights,
        "log_L_events": log_L_events,
        "inj_z": inj_z,
        "inj_m1": inj_m1,
        "inj_m2": inj_m2,
        "inj_log_pop_src": inj_log_pop_src,
        "inj_log_jac": inj_log_jac,
        "inj_log_pop_det": inj_log_pop_det,
        "log_inj_weights": log_inj_weights,
    }

    for name, x in quantities.items():
        x_np = np.asarray(x)
        finite = np.isfinite(x_np)
        print(
            f"{name:28s}",
            "shape =", x_np.shape,
            "finite =", finite.all(),
            "n_bad =", np.size(finite) - finite.sum(),
            "min =", np.nanmin(x_np),
            "max =", np.nanmax(x_np),
        )

In [ ]:
debug_values = {
    name: cfg["init"]
    for name, cfg in param_config.items()
}

debug_likelihood_at_values(debug_values)

#### Run MCMC

In [ ]:


nuts = NUTS(
    hierarchical_model,
    init_strategy=init_to_value(values=init_values),
    target_accept_prob=0.85,
)

mcmc = MCMC(
    nuts,
    num_warmup=num_warmup,
    num_samples=num_samples,
    num_chains=num_chains,
)

mcmc.run(jax.random.PRNGKey(rng_seed))
mcmc.print_summary()


## Convert to ArviZ and inspect diagnostics


In [ ]:
# Optional: to inspect a saved run later, load the tagged file explicitly, for example:
# idata = az.from_netcdf(results_dir / f"inferencedata__{run_tag}.nc")


In [ ]:
idata = az.from_numpyro(mcmc)

summary = az.summary(idata, round_to=3)
summary_path = results_dir / f"summary__{run_tag}.csv"
idata_path = results_dir / f"inferencedata__{run_tag}.nc"
posterior_path = results_dir / f"posterior_samples__{run_tag}.npz"
config_path = results_dir / f"config__{run_tag}.json"

summary.to_csv(summary_path)
idata.to_netcdf(idata_path)

posterior_flat = mcmc.get_samples(group_by_chain=False)
np.savez(
    posterior_path,
    **{name: np.asarray(value) for name, value in posterior_flat.items()},
)

def json_safe(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.generic):
        return x.item()
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, tuple):
        return [json_safe(v) for v in x]
    if isinstance(x, list):
        return [json_safe(v) for v in x]
    if isinstance(x, dict):
        return {str(k): json_safe(v) for k, v in x.items()}
    return str(x)

run_config = {
    "run_tag": run_tag,
    "eos_tag": eos_tag,
    "pop_tag": pop_tag,
    "detpe_tag": detpe_tag,
    "inj_tag": inj_tag,
    "soft_tag": soft_tag,
    "inf_tag": inf_tag,
    "input_files": {
        "eos_fit_filename": eos_fit_filename,
        "population_filename": population_filename,
        "detected_filename": detected_filename,
        "pe_filename": pe_filename,
        "injections_filename": injections_filename,
    },
    "runtime": {
        "n_events_use": n_events_use,
        "n_pe_use": n_pe_use,
        "n_injections_use": n_injections_use,
        "n_injections_available": n_injections_available,
        "inj_subset_factor": inj_subset_factor,
        "n_inj_drawn": n_inj_drawn,
        "num_warmup": num_warmup,
        "num_samples": num_samples,
        "num_chains": num_chains,
        "rng_seed": rng_seed,
        "apply_hard_eos_physical_prior": apply_hard_eos_physical_prior,
        "apply_hard_mc_variance_cut": apply_hard_mc_variance_cut,
        "max_log_likelihood_variance": max_log_likelihood_variance,
        "soft_eos_mode": soft_eos_mode,
        "sigma_soft_loglambda_fixed": sigma_soft_loglambda_fixed,
        "sigma_soft_loglambda_prior": sigma_soft_loglambda_prior,
        "sigma_soft_loglambda_init": sigma_soft_loglambda_init,
    },
    "truth": {
        "fit_min": fit_min,
        "fit_max": fit_max,
        "eos_order": eos_order,
        "eos_coeffs_fit": eos_coeffs_fit,
        "m_min_true": m_min_true,
        "m_max_true": m_max_true,
        "edge_width_true": edge_width_true,
        "alpha_z_true": alpha_z_true,
        "beta_z_true": beta_z_true,
        "z_p_true": z_p_true,
    },
    "param_config": param_config,
}

with open(config_path, "w") as f:
    json.dump(json_safe(run_config), f, indent=2)

print(f"Saved ArviZ InferenceData to {idata_path}")
print(f"Saved posterior samples to {posterior_path}")
print(f"Saved summary to {summary_path}")
print(f"Saved config to {config_path}")
summary


In [ ]:
trace_var_names = [
    name
    for name, cfg in param_config.items()
    if cfg["sample"] and name in idata.posterior
]

az.plot_trace(
    idata,
    var_names=trace_var_names,
    compact=True,
)

plt.tight_layout()
trace_path = figures_dir / f"trace__{run_tag}.png"
plt.savefig(trace_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved trace plot to {trace_path}")

## Corner plot


In [ ]:
posterior = {
    name: np.asarray(idata.posterior[name].stack(sample=("chain", "draw")).values)
    for name in idata.posterior.data_vars
}


In [ ]:
#posterior = mcmc.get_samples()

#posterior = idata.posterior

corner_names = [
    name for name, cfg in param_config.items()
    if cfg["sample"] and name in posterior
]

corner_array = np.column_stack([np.asarray(posterior[name]) for name in corner_names])

print(corner_array.shape)

fig = corner.corner(
    corner_array,
    labels=corner_names,
    truths=[param_config[name]["fixed"] for name in corner_names],
    show_titles=True,
)

corner_path = figures_dir / f"corner__{run_tag}.png"
fig.savefig(corner_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved corner plot to {corner_path}")


## Mass-distribution posterior check

In [ ]:
#posterior = mcmc.get_samples()
n_draws_plot = min(200, len(next(iter(posterior.values()))))

draw_ids = rng.choice(len(next(iter(posterior.values()))), size=n_draws_plot, replace=False)
m_plot = np.linspace(fit_min, fit_max, 400)

def bounded_smooth_mass_unnorm_np(m, m_min, m_max, edge_width, edge_pdf_value):
    logit_edge = np.log((1.0 - edge_pdf_value) / edge_pdf_value)
    edge_scale = edge_width / logit_edge

    left = 1.0 / (1.0 + np.exp(-(m - m_min - edge_width) / edge_scale))
    right = 1.0 / (1.0 + np.exp(-(m_max - edge_width - m) / edge_scale))
    return left * right

mass_pdf_draws = []

for draw_id in range(len(next(iter(posterior.values())))):
    if "m_min" in posterior:
        m_min_draw = float(posterior["m_min"][draw_id])
    else:
        m_min_draw = float(param_config["m_min"]["fixed"])

    if "m_max" in posterior:
        m_max_draw = float(posterior["m_max"][draw_id])
    else:
        m_max_draw = float(param_config["m_max"]["fixed"])

    if "edge_width" in posterior:
        edge_width_draw = float(posterior["edge_width"][draw_id])
    else:
        edge_width_draw = float(param_config["edge_width"]["fixed"])

    pdf_draw = bounded_smooth_mass_unnorm_np(
        m_plot, m_min_draw, m_max_draw, edge_width_draw, edge_pdf_value
    )
    pdf_draw /= np.trapz(pdf_draw, m_plot)
    mass_pdf_draws.append(pdf_draw)

mass_pdf_draws = np.array(mass_pdf_draws)

mass_pdf_q05 = np.percentile(mass_pdf_draws, 5, axis=0)
mass_pdf_q50 = np.percentile(mass_pdf_draws, 50, axis=0)
mass_pdf_q95 = np.percentile(mass_pdf_draws, 95, axis=0)

pdf_true = bounded_smooth_mass_unnorm_np(
    m_plot, m_min_true, m_max_true, edge_width_true, edge_pdf_value
)
pdf_true /= np.trapz(pdf_true, m_plot)

fig, ax = plt.subplots(figsize=(7.0, 4.5))

ax.fill_between(
    m_plot,
    mass_pdf_q05,
    mass_pdf_q95,
    alpha=0.25,
    label="90% posterior band",
)

for draw_id in draw_ids:
    ax.plot(m_plot, mass_pdf_draws[draw_id], lw=0.5, alpha=0.06)

ax.plot(m_plot, mass_pdf_q50, lw=1.6, label="Posterior median")
ax.plot(m_plot, pdf_true, color="black", lw=2.0, label="Injected mass PDF")
ax.axvline(m_min_true, color="black", ls="--", lw=1.0, alpha=0.7)
ax.axvline(m_max_true, color="black", ls="--", lw=1.0, alpha=0.7)

ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$p(m)$")
ax.set_title("Mass-distribution posterior")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
mass_path = figures_dir / f"mass_posterior__{run_tag}.png"
fig.savefig(mass_path, dpi=200)
plt.show()

print(f"Saved mass posterior plot to {mass_path}")

## EOS posterior check


In [ ]:
#posterior = mcmc.get_samples()
n_draws_total = len(posterior["eos_a0"])
n_draws_plot = min(200, n_draws_total)

draw_ids = rng.choice(n_draws_total, size=n_draws_plot, replace=False)
m_plot = np.linspace(fit_min, fit_max, 300)

fit_lambda = np.exp(np.polyval(eos_coeffs_fit, m_plot))

lambda_draws = []

for draw_id in range(n_draws_total):
    coeff_draw = np.array([
        posterior[f"eos_a{k}"][draw_id]
        for k in range(eos_order + 1)
    ])
    lambda_draw = np.exp(np.polyval(coeff_draw, m_plot))
    lambda_draws.append(lambda_draw)

lambda_draws = np.array(lambda_draws)

lambda_q05 = np.percentile(lambda_draws, 5, axis=0)
lambda_q50 = np.percentile(lambda_draws, 50, axis=0)
lambda_q95 = np.percentile(lambda_draws, 95, axis=0)

fig, ax = plt.subplots(figsize=(7.0, 4.5))

ax.fill_between(
    m_plot,
    lambda_q05,
    lambda_q95,
    alpha=0.25,
    label="90% posterior band",
)

for draw_id in draw_ids:
    ax.plot(m_plot, lambda_draws[draw_id], lw=0.5, alpha=0.06)

ax.plot(m_plot, lambda_q50, lw=1.6, label="Posterior median")
ax.plot(m_plot, fit_lambda, color="black", lw=2.0, label="Injected polynomial")

ax.set_yscale("log")
ax.set_ylim(0.1, 1e05)
ax.set_xlabel(r"Mass $m\,[M_\odot]$")
ax.set_ylabel(r"$\Lambda(m)$")
ax.set_title("EOS posterior with soft-constraint setting")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
eos_path = figures_dir / f"eos_posterior__{run_tag}.png"
fig.savefig(eos_path, dpi=200)
plt.show()

print(f"Saved EOS posterior plot to {eos_path}")

## Monte Carlo variance diagnostics


In [ ]:
#posterior = mcmc.get_samples()

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.5))

axes[0].hist(np.asarray(posterior["log_likelihood_variance"]), bins=40, alpha=0.7)
axes[0].axvline(max_log_likelihood_variance, color="black", ls="--")
axes[0].set_xlabel(r"$\sigma^2_{\ln\hat{\mathcal{L}}}$")
axes[0].set_ylabel("Posterior samples")
axes[0].set_title("MC variance cut")

axes[1].hist(np.asarray(posterior["sum_event_rel_var"]), bins=40, alpha=0.7)
axes[1].set_xlabel("Event contribution")
axes[1].set_title("Event MC variance")

axes[2].hist(np.asarray(posterior["selection_rel_var"]), bins=40, alpha=0.7)
axes[2].set_xlabel("Selection relative variance")
axes[2].set_title("Selection MC variance")

for ax in axes:
    ax.grid(alpha=0.3)

fig.tight_layout()
mcvar_path = figures_dir / f"mc_variance__{run_tag}.png"
fig.savefig(mcvar_path, dpi=200)
plt.show()

print(f"Saved MC variance plot to {mcvar_path}")


## Soft-constraint diagnostics

This diagnostic is only relevant when

```python
soft_eos_mode = "fixed"
```

or

```python
soft_eos_mode = "sample"
```

If the width is fixed, the plot simply reports the value being used.

If the width is sampled, the posterior distribution of

$$
\sigma_{\rm soft}
$$

is shown.

The interpretation is:

- posterior concentrated near the lower prior bound: the data prefer a nearly strict EOS constraint;

- posterior concentrated at a finite value: the strict constraint may be too sharp for the available PE samples or for the model assumptions;

- posterior drifting to large values: the model may be weakening or washing out the tidal information.

The most important comparison is with the Monte Carlo variance diagnostic.

A useful soft kernel should reduce the Monte Carlo variance without substantially changing the physical posterior for

$$
\Lambda_{\rm EOS}(m).
$$

If it changes the EOS posterior substantially, it is no longer just a numerical regularization.

In [ ]:
if "sigma_soft_loglambda" in posterior and soft_eos_mode != "strict":
    sigma_samples = np.asarray(posterior["sigma_soft_loglambda"], dtype=float)

    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    ax.hist(sigma_samples, bins=40, density=True, alpha=0.7)

    if soft_eos_mode == "sample":
        ax.axvline(sigma_soft_loglambda_prior[1], color="black", ls=":", lw=1.0, label="Prior bounds")
        ax.axvline(sigma_soft_loglambda_prior[2], color="black", ls=":", lw=1.0)
    elif soft_eos_mode == "fixed":
        ax.axvline(sigma_soft_loglambda_fixed, color="black", ls="--", lw=1.5, label="Fixed value")

    ax.set_xlabel(r"$\sigma_{\rm soft}$ in $\log\tilde\Lambda$")
    ax.set_ylabel("Posterior density")
    ax.set_title("Soft EOS kernel width")
    ax.grid(alpha=0.3)
    ax.legend()

    fig.tight_layout()
    sigma_path = figures_dir / f"soft_sigma_posterior__{run_tag}.png"
    fig.savefig(sigma_path, dpi=200)
    plt.show()

    print(f"Saved soft-sigma posterior plot to {sigma_path}")
    print(f"median sigma_soft_loglambda = {np.median(sigma_samples):.4f}")
    print(f"90% interval = [{np.quantile(sigma_samples, 0.05):.4f}, {np.quantile(sigma_samples, 0.95):.4f}]")
else:
    print("No sampled/fixed soft width to plot for strict mode.")

## DIC-like and MC-stability metrics

The metrics computed here compare the strict and soft treatments.

They are diagnostics, not exact Bayesian evidences.

The most important quantities are:

### Median log likelihood

This measures how well the model fits the detected events, after selection correction.

### DIC-like score

This is a rough goodness-of-fit score with a complexity penalty. It can be useful for comparison, but it should not be interpreted as a decisive Bayes factor.

### Monte Carlo variance

The key diagnostic for this notebook is

$$
\sigma^2_{\log\widehat{\mathcal{L}}}.
$$

If softening the EOS constraint works as intended, this quantity should decrease.

### Divergences

A reduction in NUTS divergences can indicate that the posterior geometry has become easier to sample. However, this should be interpreted together with the posterior plots: fewer divergences are not useful if the model has changed the physical inference.

The central question is:

> Does the soft EOS kernel improve numerical stability without changing the scientific conclusion?

In [ ]:
def divergence_count(idata):
    if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:
        return int(np.asarray(idata.sample_stats["diverging"]).sum())
    return 0

def dic_like_from_idata(idata):
    if "log_likelihood_value" not in idata.posterior:
        return None
    logL = np.asarray(idata.posterior["log_likelihood_value"].stack(sample=("chain", "draw")).values, dtype=float)
    logL = logL[np.isfinite(logL)]
    if len(logL) == 0:
        return None
    D = -2.0 * logL
    p_dic_var = 0.5 * np.var(D, ddof=1)
    return {
        "n_samples": len(logL),
        "median_logL": np.median(logL),
        "q05_logL": np.quantile(logL, 0.05),
        "q95_logL": np.quantile(logL, 0.95),
        "mean_deviance": np.mean(D),
        "p_dic_var": p_dic_var,
        "dic_var": np.mean(D) + p_dic_var,
    }

metrics = dic_like_from_idata(idata)
metric_row = {
    "model": soft_tag,
    "run_tag": run_tag,
    "soft_eos_mode": soft_eos_mode,
    "divergences": divergence_count(idata),
    "median_log_likelihood_variance": float(np.median(np.asarray(posterior["log_likelihood_variance"]))),
    "q95_log_likelihood_variance": float(np.quantile(np.asarray(posterior["log_likelihood_variance"]), 0.95)),
    "median_selection_rel_var": float(np.median(np.asarray(posterior["selection_rel_var"]))),
    "median_sum_event_rel_var": float(np.median(np.asarray(posterior["sum_event_rel_var"]))),
}

if metrics is not None:
    metric_row.update(metrics)

if "sigma_soft_loglambda" in posterior:
    sigma_samples = np.asarray(posterior["sigma_soft_loglambda"], dtype=float)
    metric_row.update({
        "median_sigma_soft_loglambda": float(np.median(sigma_samples)),
        "q05_sigma_soft_loglambda": float(np.quantile(sigma_samples, 0.05)),
        "q95_sigma_soft_loglambda": float(np.quantile(sigma_samples, 0.95)),
    })

soft_metrics = pd.DataFrame([metric_row])
metrics_path = results_dir / f"soft_eos_metrics__{run_tag}.csv"
soft_metrics.to_csv(metrics_path, index=False)

print(f"Saved soft-EOS metrics to {metrics_path}")
display(soft_metrics)

## Optional comparison to strict and misspecification runs

This section can load results from other notebooks and compare them with the current soft-EOS run.

The most useful comparisons are:

1. A strict matched run from notebook 03.

2. A misspecified EOS run from notebook 04.

3. The current soft-EOS run.

This comparison separates two different issues.

### Numerical sharpness of the EOS constraint

The strict and soft runs use the same global EOS model family, but different treatments of the delta-function constraint.

If the soft run mainly reduces Monte Carlo variance and leaves the EOS posterior almost unchanged, the soft kernel is acting as a numerical stabilizer.

### Global EOS model misspecification

The notebook 04 runs change the EOS model family itself.

If changing the polynomial order shifts the EOS posterior more than softening the constraint, then model misspecification is the larger source of systematic uncertainty.

If softening the constraint shifts the result by an amount comparable to changing the EOS model, then the numerical treatment of the EOS constraint is not negligible.

This comparison helps answer:

> Is the main uncertainty coming from finite-sample numerical stability, or from the assumed EOS model family?

In [ ]:
comparison_idata_filenames = {
    # Example: strict matched result from notebook 03.
    # "Strict matched": "inferencedata__<notebook03_run_tag>.nc",

    # Examples: model-order variants from notebook 04.
    # "Poly 1 misspec": "inferencedata__<notebook04_poly1_run_tag>.nc",
    # "Poly 7 misspec": "inferencedata__<notebook04_poly7_run_tag>.nc",
}

comparison_idatas = {"Current soft run": idata}

for label, filename in comparison_idata_filenames.items():
    path = results_dir / filename
    if path.exists():
        comparison_idatas[label] = az.from_netcdf(path)
        print(f"Loaded {label}: {path}")
    else:
        print(f"Skipping {label}; file not found: {path}")

rows = []
for label, idata_i in comparison_idatas.items():
    metrics_i = dic_like_from_idata(idata_i)
    if metrics_i is None:
        continue
    posterior_i = idata_i.posterior
    row = {
        "model": label,
        "divergences": divergence_count(idata_i),
        **metrics_i,
    }
    if "log_likelihood_variance" in posterior_i:
        llv = np.asarray(posterior_i["log_likelihood_variance"].stack(sample=("chain", "draw")).values, dtype=float)
        row["median_log_likelihood_variance"] = np.median(llv)
        row["q95_log_likelihood_variance"] = np.quantile(llv, 0.95)
    if "sigma_soft_loglambda" in posterior_i:
        sig = np.asarray(posterior_i["sigma_soft_loglambda"].stack(sample=("chain", "draw")).values, dtype=float)
        row["median_sigma_soft_loglambda"] = np.median(sig)
    rows.append(row)

comparison_metrics = pd.DataFrame(rows)
if len(comparison_metrics) > 0:
    if "Strict matched" in set(comparison_metrics["model"]):
        ref = comparison_metrics.loc[comparison_metrics["model"] == "Strict matched"].iloc[0]
    else:
        ref = comparison_metrics.iloc[0]
    comparison_metrics["delta_median_logL_vs_ref"] = comparison_metrics["median_logL"] - ref["median_logL"]
    comparison_metrics["delta_dic_var_vs_ref"] = comparison_metrics["dic_var"] - ref["dic_var"]

    comparison_path = results_dir / f"soft_eos_cross_comparison__{run_tag}.csv"
    comparison_metrics.to_csv(comparison_path, index=False)
    print(f"Saved cross-comparison metrics to {comparison_path}")
    display(comparison_metrics)
else:
    print("No comparison metrics available.")

## Interpretation

Use this notebook to answer a numerical question:

> Is the strict delta-function EOS constraint stable when the event likelihood is represented by finite PE samples?

The strict model is physically natural for a universal EOS,

$$
\Lambda=\Lambda_{\rm EOS}(m).
$$

But numerically it can be brittle because a delta-function relation defines a lower-dimensional surface in the full tidal parameter space.

The soft model replaces that exact surface by a finite-width kernel,

$$
p_{\rm soft}(\log\tilde{\Lambda}|m_1,m_2,\phi_{\rm EOS},\sigma_{\rm soft})
=
\mathcal{N}
\left[
\log\tilde{\Lambda}
\middle|
\log\tilde{\Lambda}_{\rm EOS},
\sigma_{\rm soft}
\right].
$$

This can reduce Monte Carlo variance by allowing more PE samples to contribute to the event likelihood.

However, the price is that the tidal constraint is weakened:

$$
\sigma_{\rm eff,i}^2
=
\sigma_{\ell,i}^2+\sigma_{\rm soft}^2.
$$

Therefore the soft kernel must be interpreted carefully.

## Questions to answer

1. Does the soft kernel reduce

$$
\sigma^2_{\log\widehat{\mathcal{L}}}?
$$

2. Does it reduce NUTS divergences or improve effective sample size?

3. Does it leave the EOS posterior band essentially unchanged?

4. If $\sigma_{\rm soft}$ is sampled, does the posterior prefer a value close to zero or a finite width?

5. Are the shifts caused by the soft constraint smaller than, comparable to, or larger than the shifts caused by EOS model misspecification in notebook 04?

## Main takeaway

A soft EOS constraint can be useful as a numerical regularization of the strict delta-function treatment.

But it is not, by itself, a solution to the physical question:

> What if the global EOS model is wrong?

That question was the focus of notebook 04.

Notebook 05 instead teaches that even when the EOS model family is fixed, the numerical implementation of a deterministic EOS relation can matter when likelihoods are estimated with finite Monte Carlo samples.